# LifeMate — đo thời gian inference
ASR PhoWhisper-large (e46f) → bộ lọc app BM25 ∪ embedding → FC Qwen3.5-4B (vLLM cục bộ) → ToolEnv.
Kết quả: `/result/jupyter_submission.jsonl` (cùng định dạng `submission.jsonl`, có trường `time`).

In [ ]:
# Bước 1: set seed cố định
import os, sys, random
sys.path.insert(0, "/code/src")
import pipeline as P

def seed_everything(seed=42):
    import numpy as np, torch
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [ ]:
# Bước 2: nạp mô hình và tài nguyên (không dùng API ngoài; vLLM chạy cục bộ trong container)
import json, time
from pathlib import Path

ROOT = Path("/data")
ROOT, SPLIT = P.find_split(ROOT)
t0 = time.perf_counter()
pipe = P.Pipeline(ROOT, SPLIT)      # vLLM (FC) + ASR + bộ lọc app + tokenizer
print(f"load model: {(time.perf_counter() - t0) * 1000:.0f} ms")

In [ ]:
# Bước 3: đọc test case
sys.path.insert(0, str(ROOT / SPLIT))   # zaic_tools của split
from zaic_tools import ToolEnv

with open(ROOT / SPLIT / f"{SPLIT}.jsonl", encoding="utf-8") as f:
    test_cases = [json.loads(line) for line in f if line.strip()]
print(SPLIT, len(test_cases), "mẫu")

In [ ]:
# Bước 4: dự đoán từng mẫu, đo thời gian (ToolEnv tạo trong pipe.run_sample: đúng một ToolEnv cho mỗi mẫu)
all_result = []
try:
    for item in test_cases:
        t1 = time.perf_counter()
        tool_calls, state_hash, _, _ = pipe.run_sample(item)     # ASR -> lọc app -> LLM -> env.call(...) -> ...
        ms = int((time.perf_counter() - t1) * 1000)
        all_result.append({"id": item["id"], "tool_calls": tool_calls, "state_hash": state_hash, "time": ms})
finally:
    pipe.close()

Path("/result").mkdir(parents=True, exist_ok=True)
with open("/result/jupyter_submission.jsonl", "w", encoding="utf-8") as f:
    for r in all_result:
        f.write(json.dumps(r, ensure_ascii=False) + "
")
times = [r["time"] for r in all_result]
print(f"{len(times)} mẫu, trung bình {sum(times) / max(len(times), 1):.0f} ms/mẫu, max {max(times or [0])} ms")